In [19]:
import torch
import optuna
import pandas as pd
from torch_geometric.datasets import QM9
from torch_geometric.loader import DataLoader
from sklearn.model_selection import train_test_split

# ====== 你之前写的工具函数 ======
from train import run_training, evaluate, compute_task_stats, WMAELoss
from model import WDMPNNModel   # 就是你写的带 adapter 的 model.py

In [20]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [21]:
print(device)

cuda


In [22]:
# -------------------- 参数搜索空间 --------------------
def suggest_params(trial):
    return {
        # === Encoder ===
        "hidden_dim": trial.suggest_categorical("hidden_dim", [128, 256, 512]),
        "num_layers": trial.suggest_int("num_layers", 2, 5),
        "act": trial.suggest_categorical("act", ["relu", "silu", "mish"]),
        "dropout": trial.suggest_float("dropout", 0.0, 0.5),

        # === Attention ===
        "use_edge_attn": trial.suggest_categorical("use_edge_attn", [True, False]),
        "att_hidden": trial.suggest_int("att_hidden", 32, 128),

        # === Pooling ===
        "pool": trial.suggest_categorical("pool", ["mean", "att"]),

        # === Adapter ===
        "adapter_kind": trial.suggest_categorical("adapter_kind", ["none", "linear", "mlp"]),
        "adapter_hidden": trial.suggest_int("adapter_hidden", 16, 128),
        "adapter_dropout": trial.suggest_float("adapter_dropout", 0.0, 0.3),

        # === Head ===
        "mlp_hidden": trial.suggest_categorical("mlp_hidden", [
            (128, 64),
            (256, 128),
            (256, 128, 64),
        ]),
        "head_dropout": trial.suggest_float("head_dropout", 0.0, 0.5),

        # === Optimizer ===
        "lr_encoder": trial.suggest_float("lr_encoder", 1e-5, 5e-4, log=True),
        "lr_adapter": trial.suggest_float("lr_adapter", 1e-4, 1e-3, log=True),
        "lr_head": trial.suggest_float("lr_head", 1e-4, 1e-3, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True),

        # === Training ===
        "batch_size": trial.suggest_categorical("batch_size", [32, 64, 128]),
    }


# -------------------- 数据加载 --------------------
def load_qm9(batch_size=64, num_workers=0):
    dataset = QM9(root="kaggle/input/my-qm9/qm9")
    idx = list(range(len(dataset)))
    train_idx, val_idx = train_test_split(idx, test_size=0.1, random_state=42)

    train_ds = dataset[train_idx]
    val_ds = dataset[val_idx]

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers)
    
    QM9_TASKS = [
    "mu", "alpha", "homo", "lumo", "gap", "r2", "zpve",
    "U0", "U", "H", "G", "Cv",
    "u0_atom", "u_atom", "h_atom", "g_atom",
    "A", "B", "C",
    ]

    # 转换成 DataFrame 方便统计 n_dict / r_dict
    y = dataset._data.y.numpy()
    df = pd.DataFrame(y, columns=QM9_TASKS)

    return train_loader, val_loader, df, QM9_TASKS, dataset


# -------------------- Optuna 目标函数 --------------------
def objective(trial):
    params = suggest_params(trial)

    # === 数据 ===
    train_loader, val_loader, df, tasks, dataset = load_qm9(batch_size=params["batch_size"])
    node_dim = dataset.num_node_features
    edge_dim = dataset.num_edge_features

    # === 模型 ===
    model = WDMPNNModel(
        node_dim=node_dim,
        edge_dim=edge_dim,
        hidden_dim=params["hidden_dim"],
        num_layers=params["num_layers"],
        tasks=tasks,
        mlp_hidden=list(params["mlp_hidden"]),
        use_edge_attn=params["use_edge_attn"],
        dropout=params["dropout"],
        act=params["act"],
        pool=params["pool"],
        adapter_kind=params["adapter_kind"],
        adapter_hidden=params["adapter_hidden"],
        adapter_dropout=params["adapter_dropout"],
    ).to(device)

    # === Optimizer ===
    groups = model.param_groups()
    optimizer = torch.optim.Adam([
        {"params": groups["encoder"], "lr": params["lr_encoder"], "weight_decay": params["weight_decay"]},
        {"params": groups["adapter"], "lr": params["lr_adapter"], "weight_decay": params["weight_decay"]},
        {"params": groups["head"], "lr": params["lr_head"], "weight_decay": params["weight_decay"]},
    ])

    # === 训练 ===
    model, history = run_training(
        model,
        train_loader,
        val_loader,
        optimizer,
        tasks,
        df,
        device=device,
        max_epochs=30,
        patience=10,
    )
    for h in history:
        print(f"[Trial {trial.number}] Epoch {h['epoch']}: "
            f"Train={h['train_loss']:.4f}, Val={h['val_loss']:.4f}")

    # === 验证集总 loss ===
    n_dict, r_dict = compute_task_stats(df, tasks)
    loss_fn = WMAELoss(tasks, n_dict, r_dict)
    val_loss, _ = evaluate(model, val_loader, loss_fn, device, tasks)

    # === 保存权重 ===
    save_path = f"checkpoints/trial_{trial.number}.pt"
    torch.save(model.state_dict(), save_path)
    print(f"[Trial {trial.number}] Saved best model to {save_path} with val_loss={val_loss:.4f}")

    return val_loss

In [23]:
# -------------------- 主入口 --------------------
if __name__ == "__main__":
    study = optuna.create_study(
        study_name="qm9_pretrain_study",
        storage="sqlite:///optuna_qm9_pretrain.db",
        load_if_exists=True,
        direction="minimize",
        )
    study.optimize(objective, n_trials=50)

    print("Best trial:", study.best_trial.params)
    print("Best val_loss:", study.best_trial.value)

[I 2025-09-10 09:04:02,739] Using an existing study with name 'qm9_pretrain_study' instead of creating a new one.
/scratch/e1350261/venvs/ai39/lib64/python3.9/site-packages/optuna/distributions.py:518: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains [128, 64] which is of type list.
  warnings.warn(message)
/scratch/e1350261/venvs/ai39/lib64/python3.9/site-packages/optuna/distributions.py:518: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains [256, 128] which is of type list.
  warnings.warn(message)
/scratch/e1350261/venvs/ai39/lib64/python3.9/site-packages/optuna/distributions.py:518: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains [256, 128, 64] which is of type list.
  warnings.warn(message)
/scratch/e1350261/venv

Epoch 001: Train=0.2264, Val=0.2297 || mu: 1.1762 | alpha: 22.0857 | homo: 0.9478 | lumo: 1.0556 | gap: 1.5134 | r2: 1016.7528 | zpve: 0.9303 | U0: 11016.8286 | U: 11012.7530 | H: 11020.4072 | G: 11015.3601 | Cv: 8.6821 | u0_atom: 22.0491 | u_atom: 22.1724 | h_atom: 22.4183 | g_atom: 20.5654 | A: 1.3714 | B: 0.3882 | C: 0.2630
Epoch 002: Train=0.1789, Val=0.1972 || mu: 1.1921 | alpha: 20.0259 | homo: 0.4728 | lumo: 1.1423 | gap: 1.2636 | r2: 257.4158 | zpve: 0.8932 | U0: 9933.3327 | U: 9917.7621 | H: 9945.6244 | G: 9919.3162 | Cv: 7.5586 | u0_atom: 20.7663 | u_atom: 20.8877 | h_atom: 21.0164 | g_atom: 19.2952 | A: 1.2150 | B: 0.3781 | C: 0.2882
Epoch 003: Train=0.1075, Val=0.1678 || mu: 1.1616 | alpha: 28.4674 | homo: 0.9422 | lumo: 1.2651 | gap: 2.0901 | r2: 472.5650 | zpve: 1.2853 | U0: 4470.8849 | U: 4460.8321 | H: 4479.1378 | G: 4461.2730 | Cv: 10.8299 | u0_atom: 27.8294 | u_atom: 28.1785 | h_atom: 28.0765 | g_atom: 25.8800 | A: 1.6283 | B: 0.3801 | C: 0.2954


[W 2025-09-10 09:05:46,919] Trial 5 failed with parameters: {'hidden_dim': 512, 'num_layers': 3, 'act': 'silu', 'dropout': 0.36319259516853614, 'use_edge_attn': True, 'att_hidden': 84, 'pool': 'att', 'adapter_kind': 'linear', 'adapter_hidden': 46, 'adapter_dropout': 0.21806807316529947, 'mlp_hidden': (256, 128), 'head_dropout': 0.22623013321659896, 'lr_encoder': 1.1013667901610914e-05, 'lr_adapter': 0.0003394317184109074, 'lr_head': 0.00016637454228824177, 'weight_decay': 3.0663856098283634e-05, 'batch_size': 64} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/scratch/e1350261/venvs/ai39/lib64/python3.9/site-packages/optuna/study/_optimize.py", line 201, in _run_trial
    value_or_values = func(trial)
  File "/var/tmp/pbs.208121.stdct-mgmt-02/ipykernel_1749259/4039901976.py", line 102, in objective
    model, history = run_training(
  File "/nfs/home/svu/e1350261/kaggle/my_wdmpnn/train.py", line 154, in run_training
    train_loss, train

KeyboardInterrupt: 